# 03 · Quantize int8 + so sánh 1 tầng vs 2 tầng + đo tốc độ CPU

Chạy **CPU** (Kaggle 4 nhân, Colab free 2 nhân) - cố ý không dùng GPU để số đo gần với laptop.
Kaggle: Settings → Accelerator → None (CPU). Colab: runtime CPU mặc định là được.
Input: code + VNTS + output notebook 01 (models/detector_*.onnx, datasets) + 02 (models/signnet*.onnx).
Trên Colab các file này đã nằm trong Drive sau khi chạy 01, 02.
Secret `HF_TOKEN` để đẩy model lên HF Hub (Kaggle: Add-ons → Secrets; Colab: biểu tượng chìa khoá bên trái).

In [ ]:
import glob, json, os, shutil, sys

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-dashcam-vision" if IN_COLAB else os.path.abspath("vn-dashcam-vision")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-dashcam-vision {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-dashcam-vision" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


# Colab: đọc hàng nghìn ảnh từ Google Drive rất chậm -> dataset để ở ổ cục bộ /content (mất khi ngắt,
# chạy lại cell là có), còn model / kết quả / checkpoint YOLO thì nằm trong WORK trên Drive.
DATA = "/content/data" if IN_COLAB else "datasets"
DS = f"{DATA}/vnts"
os.makedirs(DATA, exist_ok=True)

# VNTS: Kaggle add dataset làm input; Colab tải bằng kaggle CLI về ổ cục bộ
if IN_KAGGLE:
    VNTS = [d for d in glob.glob("/kaggle/input/*") if "traffic" in d.lower()][0]
else:
    VNTS = f"{DATA}/vnts_raw"
    if not glob.glob(f"{VNTS}/*"):
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            os.environ[k] = secret(k) or ""
        !kaggle datasets download -d maitam/vietnamese-traffic-signs -p {VNTS} --unzip
os.makedirs("models", exist_ok=True)
if IN_KAGGLE:
    for f in glob.glob("/kaggle/input/**/models/*", recursive=True):
        shutil.copy(f, "models/")
else:  # Colab: models/ của notebook 01, 02 đã nằm trong WORK
    find("models/detector_1cls.onnx", "chạy notebook 01 trước")
    find("models/signnet.onnx", "chạy notebook 02 trước")
!pip install -q onnx onnxruntime opencv-python-headless
if IN_KAGGLE or not os.path.exists(f"{DS}/yolo_ncls/data.yaml"):
    !python -m dashcam.prepare_vnts --src {VNTS} --out {DS}
!ls -la models

In [ ]:
!python -m dashcam.export --yolo models/detector_1cls.onnx --yolo models/detector_ncls.onnx \
    --calib-images {DS}/yolo_ncls/images/train --imgsz 640 --out models

## mAP@0.5 trên test (ảnh tĩnh) + ms/ảnh với 4 luồng CPU

In [ ]:
!python -m dashcam.evaluate --images {DS}/yolo_ncls/images/test --labels {DS}/yolo_ncls/labels/test \
    --models models --class-stats {DS}/class_stats.json --threads 4 --limit 400 --out results
print(open("results/eval.md", encoding="utf-8").read())

## 1 luồng (giống thiết bị nhúng yếu)

In [ ]:
!python -m dashcam.evaluate --images {DS}/yolo_ncls/images/test --labels {DS}/yolo_ncls/labels/test \
    --models models --threads 1 --limit 150 --out results_1thread
print(open("results_1thread/eval.md", encoding="utf-8").read())

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=secret("HF_TOKEN"))
REPO = "DuongCodeAI/vn-dashcam-vision"
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="models", repo_id=REPO, allow_patterns=["*.onnx", "*.json"])
api.upload_folder(folder_path="results", repo_id=REPO, path_in_repo="results")
print(f"https://huggingface.co/{REPO}")